In [ ]:
import logging
from pyspark.sql import functions as F

# ==============================================================================
# LOGGING
# ==============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] - %(message)s",
    handlers=[logging.StreamHandler()]
)

logger = logging.getLogger("consulta-dados")

logger.info("=" * 80)
logger.info("INICIANDO PROJETO DE CONSULTA")
logger.info("=" * 80)

# ==============================================================================
# CAPTURA DOS PARÂMETROS
# ==============================================================================

# 🚀 CORRIGIDO: Default setado para o catálogo gerado pela carga para permitir testes interativos sem quebras
dbutils.widgets.text("catalog_name", "catalog_carga_dev")

catalog = dbutils.widgets.get("catalog_name").strip()

logger.info("Widgets recebidos:")
logger.info(dbutils.widgets.getAll())

if not catalog:
    raise Exception("Parâmetro catalog_name não foi recebido pelo Job.")

logger.info("=" * 80)
logger.info(f"🎯 Catálogo recebido: {catalog}")
logger.info("=" * 80)

# ==============================================================================
# VALIDAÇÃO DO CATÁLOGO
# ==============================================================================

logger.info("Listando catálogos disponíveis...")

spark.sql("SHOW CATALOGS").show(truncate=False)

spark.sql(f"USE CATALOG {catalog}")

catalogo_ativo = spark.sql("SELECT current_catalog()").collect()[0][0]

logger.info(f"✅ Catálogo ativo: {catalogo_ativo}")

logger.info("Schemas disponíveis:")

spark.sql("SHOW SCHEMAS").show(truncate=False)

In [ ]:
logger.info("=" * 80)
logger.info("LEITURA DAS TABELAS")
logger.info("=" * 80)

table_students = f"{catalog}.student.tbl_student_info"
table_courses = f"{catalog}.course.tbl_course_info"

logger.info(f"Tabela estudantes: {table_students}")
logger.info(f"Tabela cursos     : {table_courses}")

try:

    logger.info("Carregando tabela de estudantes...")

    df_students = spark.read.table(table_students)

    students_count = df_students.count()

    logger.info(
        f"✅ Estudantes carregados com sucesso. Registros: {students_count}"
    )

    logger.info("Carregando tabela de cursos...")

    df_courses = spark.read.table(table_courses)

    courses_count = df_courses.count()

    logger.info(
        f"✅ Cursos carregados com sucesso. Registros: {courses_count}"
    )

    logger.info("Prévia da tabela estudantes")

    df_students.show(10, truncate=False)

    logger.info("Prévia da tabela cursos")

    df_courses.show(10, truncate=False)

except Exception as e:
    logger.error("❌ Falha ao ler tabelas")
    logger.error(str(e))
    raise


In [ ]:
logger.info("=" * 80)
logger.info("PROCESSAMENTO")
logger.info("=" * 80)

try:

    df_dashboard = (
        df_students
        .join(
            df_courses,
            df_students.course_id == df_courses.course_id,
            "inner"
        )
        .groupBy(
            df_courses.course_name
        )
        .agg(
            F.count(df_students.student_id).alias("total_alunos")
        )
        .orderBy(
            F.desc("total_alunos")
        )
    )

    total_linhas = df_dashboard.count()

    logger.info(
        f"✅ Dashboard gerado com sucesso. Registros: {total_linhas}"
    )

except Exception as e:
    logger.error("❌ Falha durante o processamento")
    logger.error(str(e))
    raise


In [ ]:
logger.info("=" * 80)
logger.info("RESULTADO FINAL")
logger.info("=" * 80)

df_dashboard.show(truncate=False)

display(df_dashboard)

logger.info("✅ Consulta concluída com sucesso")
logger.info(f"✅ Catálogo utilizado: {catalog}")
logger.info(f"✅ Tabela origem alunos: {table_students}")
logger.info(f"✅ Tabela origem cursos: {table_courses}")
logger.info("=" * 80)